In [7]:
%%writefile cpu_worker.py
def cpu_task(n):
    s = 0
    for i in range(n):
        s += i * i
    return s

Writing cpu_worker.py


In [13]:
import time
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor,ThreadPoolExecutor
from cpu_worker import cpu_task
if __name__ == "__main__":
    mp.freeze_support()
    N = 5_000_000
    t0 = time.perf_counter()
    [cpu_task(N) for _ in range(4)]
    print("串行:",round(time.perf_counter()-t0,3))
    t0 = time.perf_counter()
    with ThreadPoolExecutor(4) as ex:
        list(ex.map(cpu_task,[N] * 4))
    print("线程:",round(time.perf_counter()-t0,3))
    t0 = time.perf_counter()
    with ProcessPoolExecutor(4) as ex:
        list(ex.map(cpu_task,[N] * 4))
    print("进程:",round(time.perf_counter()-t0,3))

串行: 0.671
线程: 0.651
进程: 0.384


In [22]:
import threading
import time 
def worker(name,delay):
    print(f"线程{name}开始执行")
    time.sleep(delay)
    print(f"线程{name}执行结束")
t = threading.Thread(target=worker,args=("A",1),daemon=False)
t.start()
t.join(timeout=2)
print("主线程继续")
d = threading.Thread(target=worker,args=("守护",5),daemon=True)
d.start()
time.sleep(1)
print("主线程结束，守护线程也会结束")
threads = []
for i in range(5):
    t = threading.Thread(target=worker,args=(f"线程{i}",2))
    t.start()
    threads.append(t)
for t in threads:
    t.join()
print("全部完成")
from concurrent.futures import ThreadPoolExecutor,as_completed
def fetch(url):
    time.sleep(0.3)
    return f"{url} -> ok"
urls = [f"http://api/{i}" for i in range(6)]
with ThreadPoolExecutor(max_workers=3) as ex:
    futures = [ex.submit(fetch,u) for u in urls]
    for f in as_completed(futures):
        try:
            print(f.result())
        except Exception as e:
            print("任务失败:",e)
with ThreadPoolExecutor(3) as ex:
    for url,res in zip(urls,ex.map(fetch,urls)):
        print(url,res)

线程A开始执行
线程A执行结束
主线程继续
线程守护开始执行
主线程结束，守护线程也会结束
线程线程0开始执行
线程线程1开始执行
线程线程2开始执行
线程线程3开始执行
线程线程4开始执行
线程线程0执行结束线程线程1执行结束
线程线程2执行结束
线程线程3执行结束
线程线程4执行结束

全部完成
http://api/0 -> ok
http://api/2 -> ok
http://api/1 -> ok
http://api/4 -> ok
http://api/3 -> ok
http://api/5 -> ok
http://api/0 http://api/0 -> ok
http://api/1 http://api/1 -> ok
http://api/2 http://api/2 -> ok
http://api/3 http://api/3 -> ok
http://api/4 http://api/4 -> ok
http://api/5 http://api/5 -> ok


线程守护执行结束


In [28]:
counter = 0
def inc():
    global counter
    for _ in range(100_000):
        counter += 1
threads = [threading.Thread(target=inc) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(counter)
counter = 0
lock = threading.Lock()
def inc_safe():
    global counter
    for _ in range(100_000):
        with lock:
            counter += 1
threads = [threading.Thread(target=inc_safe) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(counter)

400000
400000


In [32]:
import threading
rlock = threading.RLock()
class Account:
    def __init__(self,name,balance):
        self.name,self.balance = name,balance
        self._lock = rlock.__class__()
    def withdraw(self,amount):
        with self._lock:
            if self.balance >= amount:
                self.balance -= amount
                return True
            return False
    def transfer(self,other,amount):
        with self._lock:
            if self.withdraw(amount):
                other.deposit(amount)
                return True
        return False
    def deposit(self,amount):
        with self._lock:
            self.balance += amount    
alice = Account('Alice',1000)
bob =  Account('Bob',500)
print(f"初始:{alice.name}={alice.balance},{bob.name}={bob.balance}")
alice.withdraw(200)
print(f"Alice取款后:{alice.balance}")

初始:Alice=1000,Bob=500
Alice取款后:800
